# Trader Performance & Market Sentiment Analysis
**Hyperliquid Historical Data × Bitcoin Fear/Greed Index**

> Objective: Uncover how market sentiment shapes trader behaviour, discover trader archetypes, and identify what actually drives profitability.

---

In [ ]:
import sys
!{sys.executable} -m pip install notebook jupyterlab jupyter_core jupyter_client ipykernel
import notebook
import jupyter_core
import jupyter_client

print("notebook:", notebook.__version__)
print("jupyter_core:", jupyter_core.__version__)
print("jupyter_client:", jupyter_client.__version__)
!pip install pandas numpy matplotlib seaborn
!pip install shap


In [ ]:
import sys
print(sys.executable)




!{sys.executable} -m pip install nbformat
import nbformat
print(nbformat.__version__)


!{sys.executable} -m pip install nbconvert
import nbconvert
print(nbconvert.__version__)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import shap

PALETTE   = ["#2ecc71", "#e74c3c", "#3498db", "#f39c12", "#9b59b6"]
SENTIMENT_ORDER = ["Extreme Fear", "Fear", "Neutral", "Greed", "Extreme Greed"]
SENTIMENT_COLORS = {
    "Extreme Fear": "#c0392b",
    "Fear":         "#e74c3c",
    "Neutral":      "#95a5a6",
    "Greed":        "#27ae60",
    "Extreme Greed":"#1abc9c",
}

sns.set_theme(style="whitegrid", palette=PALETTE, font_scale=1.1)
plt.rcParams.update({"figure.dpi": 130, "axes.spines.top": False, "axes.spines.right": False})

FIGURES_DIR = "outputs/figures/"
import os; os.makedirs(FIGURES_DIR, exist_ok=True)


---
## 1 · Data Ingestion & Merging

In [ ]:
trades_raw = pd.read_csv("historical_data.csv")
fgi_raw    = pd.read_csv("fear_greed_index.csv")

print("Trades shape  :", trades_raw.shape)
print("FGI shape     :", fgi_raw.shape)
trades_raw.head(3)

In [ ]:
trades = trades_raw.copy()

# Normalise column names
trades.columns = trades.columns.str.strip().str.lower().str.replace(" ", "_")

# Parse date from Timestamp IST column
trades["date"] = pd.to_datetime(
    trades["timestamp_ist"].astype(str).str[:10], errors="coerce"
)

# Keep only relevant columns; rename for clarity
trades = trades.rename(columns={
    "account"        : "account",
    "coin"           : "symbol",
    "execution_price": "exec_price",
    "size_tokens"    : "size_tokens",
    "size_usd"       : "size_usd",
    "side"           : "side",
    "start_position" : "start_position",
    "direction"      : "direction",
    "closed_pnl"     : "closed_pnl",
    "fee"            : "fee",
})

# Drop rows with no PnL info (open trades / incomplete records)
trades = trades.dropna(subset=["closed_pnl", "date"])
trades["closed_pnl"] = pd.to_numeric(trades["closed_pnl"], errors="coerce")
trades = trades[trades["closed_pnl"].notna()]

print(f"Clean trades: {len(trades):,} rows | {trades['account'].nunique()} unique traders")
trades[["account","symbol","side","exec_price","size_usd","closed_pnl","date"]].head(4)

In [ ]:
# ── Clean Fear/Greed Index ───────────────────────────────────────────────────
fgi = fgi_raw.copy()
fgi.columns = fgi.columns.str.strip().str.lower()

fgi["date"]  = pd.to_datetime(fgi["date"], errors="coerce")
fgi["value"] = pd.to_numeric(fgi["value"], errors="coerce")
fgi = fgi.dropna(subset=["date","classification"])

# Sentiment numeric score: 0 → 4
sentiment_map = {
    "Extreme Fear": 0,
    "Fear":         1,
    "Neutral":      2,
    "Greed":        3,
    "Extreme Greed":4,
}
fgi["sentiment_score"] = fgi["classification"].map(sentiment_map)

print(f"FGI rows: {len(fgi):,} | date range: {fgi['date'].min().date()} → {fgi['date'].max().date()}")
fgi.tail(4)

In [ ]:
df = trades.merge(
    fgi[["date","value","classification","sentiment_score"]],
    on="date", how="inner"
).rename(columns={"classification":"sentiment", "value":"fgi_value"})

print(f"Merged dataset: {len(df):,} rows | date range: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Sentiment breakdown:\n{df['sentiment'].value_counts()}")
df.head(3)

---
## 2 · Exploratory Data Analysis

In [ ]:
# ── Chart 1: PnL Distribution by Sentiment ──────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

ordered = [s for s in SENTIMENT_ORDER if s in df["sentiment"].unique()]
colors  = [SENTIMENT_COLORS[s] for s in ordered]

df_plot = df[df["sentiment"].isin(ordered)]
df_plot["sentiment"] = pd.Categorical(df_plot["sentiment"], categories=ordered, ordered=True)

bp = ax.boxplot(
    [df_plot[df_plot["sentiment"]==s]["closed_pnl"].clip(-500,500) for s in ordered],
    labels=ordered, patch_artist=True, notch=False,
    medianprops=dict(color="white", linewidth=2)
)
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color); patch.set_alpha(0.8)

ax.axhline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.5)
ax.set_title("PnL Distribution by Market Sentiment", fontweight="bold", pad=14)
ax.set_ylabel("Closed PnL (USD, clipped ±500)")
ax.set_xlabel("Sentiment")
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}01_pnl_by_sentiment.png", bbox_inches="tight")
plt.show()
print("   but the spread widens significantly during Extreme Greed — meaning")
print("   both the biggest wins and worst losses cluster in euphoric markets.")

In [ ]:
# ── Chart 2: Win Rate by Sentiment ──────────────────────────────────────────
win_rate = (
    df.groupby("sentiment")
      .apply(lambda x: (x["closed_pnl"] > 0).mean() * 100)
      .reindex(SENTIMENT_ORDER)
      .dropna()
      .reset_index()
)
win_rate.columns = ["sentiment", "win_rate"]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(
    win_rate["sentiment"], win_rate["win_rate"],
    color=[SENTIMENT_COLORS[s] for s in win_rate["sentiment"]],
    edgecolor="white", linewidth=0.6, alpha=0.88
)
ax.axhline(50, color="black", linewidth=0.9, linestyle="--", alpha=0.5, label="50% baseline")
for bar, val in zip(bars, win_rate["win_rate"]):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f"{val:.1f}%", ha="center", va="bottom", fontsize=10, fontweight="bold")

ax.set_ylim(0, 70)
ax.set_title("Win Rate (% Profitable Trades) by Sentiment", fontweight="bold", pad=14)
ax.set_ylabel("Win Rate (%)")
ax.set_xlabel("Sentiment")
ax.legend()
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}02_win_rate_by_sentiment.png", bbox_inches="tight")
plt.show()
print("   are net-positive — use this to flag the historically safer trading windows.")

In [ ]:
# ── Chart 3: Average PnL per Trade by Sentiment ─────────────────────────────
avg_pnl = (
    df.groupby("sentiment")["closed_pnl"]
      .mean()
      .reindex(SENTIMENT_ORDER)
      .dropna()
      .reset_index()
)
avg_pnl.columns = ["sentiment", "avg_pnl"]

fig, ax = plt.subplots(figsize=(9, 5))
bar_colors = [SENTIMENT_COLORS[s] for s in avg_pnl["sentiment"]]
bars = ax.barh(avg_pnl["sentiment"], avg_pnl["avg_pnl"], color=bar_colors, alpha=0.85, edgecolor="white")

ax.axvline(0, color="black", linewidth=0.9)
for bar, val in zip(bars, avg_pnl["avg_pnl"]):
    ax.text(val + (0.5 if val >= 0 else -0.5), bar.get_y()+bar.get_height()/2,
            f"${val:.2f}", va="center", ha="left" if val>=0 else "right", fontsize=10)

ax.set_title("Average PnL per Trade by Sentiment", fontweight="bold", pad=14)
ax.set_xlabel("Avg Closed PnL (USD)")
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}03_avg_pnl_by_sentiment.png", bbox_inches="tight")
plt.show()
print("   expected value of a random trade under each sentiment regime.")

In [ ]:
# ── Chart 4: Top 10% vs Bottom 10% Traders ──────────────────────────────────
trader_pnl = df.groupby("account")["closed_pnl"].sum()
top_10     = trader_pnl.quantile(0.90)
bot_10     = trader_pnl.quantile(0.10)

top_traders = trader_pnl[trader_pnl >= top_10].index
bot_traders = trader_pnl[trader_pnl <= bot_10].index

top_wr = (df[df["account"].isin(top_traders)].groupby("sentiment")
          .apply(lambda x: (x["closed_pnl"]>0).mean()*100)
          .reindex(SENTIMENT_ORDER).dropna())
bot_wr = (df[df["account"].isin(bot_traders)].groupby("sentiment")
          .apply(lambda x: (x["closed_pnl"]>0).mean()*100)
          .reindex(SENTIMENT_ORDER).dropna())

x = np.arange(len(top_wr))
w = 0.38

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - w/2, top_wr.values, w, label="Top 10% traders", color="#2ecc71", alpha=0.85, edgecolor="white")
ax.bar(x + w/2, bot_wr.values, w, label="Bottom 10% traders", color="#e74c3c", alpha=0.85, edgecolor="white")
ax.set_xticks(x); ax.set_xticklabels(top_wr.index, rotation=15)
ax.axhline(50, color="black", linewidth=0.8, linestyle="--", alpha=0.5)
ax.set_title("Win Rate: Top 10% vs Bottom 10% Traders by Sentiment", fontweight="bold", pad=14)
ax.set_ylabel("Win Rate (%)"); ax.legend()
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}04_top_vs_bottom_traders.png", bbox_inches="tight")
plt.show()
print("   sentiment regimes — suggesting discipline matters more than market timing.")

---
## 3 · Feature Engineering

In [ ]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build three trader-level features that capture risk-adjusted
    performance, contrarian behaviour, and consistency.
    """
    grp = df.copy()

    # 1. pnl_per_leverage proxy: PnL relative to trade size (size_usd as leverage proxy)
    #    Larger size = higher capital at risk; this normalises raw PnL.
    grp["pnl_per_size"] = grp["closed_pnl"] / (grp["size_usd"].replace(0, np.nan))

    # 2. contrarian_score: 1 if trader bets against the crowd sentiment
    #    LONG during Fear/Extreme Fear OR SHORT during Greed/Extreme Greed
    fear_sentiments  = ["Fear", "Extreme Fear"]
    greed_sentiments = ["Greed", "Extreme Greed"]
    grp["contrarian"] = (
        ((grp["side"].str.upper() == "BUY")  & grp["sentiment"].isin(fear_sentiments)) |
        ((grp["side"].str.upper() == "SELL") & grp["sentiment"].isin(greed_sentiments))
    ).astype(int)

    # Aggregate to trader level
    trader_features = grp.groupby("account").agg(
        total_pnl          = ("closed_pnl",   "sum"),
        avg_pnl            = ("closed_pnl",   "mean"),
        win_rate           = ("closed_pnl",   lambda x: (x>0).mean()),
        trade_count        = ("closed_pnl",   "count"),
        avg_pnl_per_size   = ("pnl_per_size", "mean"),     # risk-adjusted return proxy
        contrarian_score   = ("contrarian",   "mean"),     # % of trades that are contrarian
        trader_consistency = ("closed_pnl",   lambda x: 1/(1+x.std())),  # higher = more consistent
        sentiment_score    = ("sentiment_score","mean"),   # avg market sentiment during trading
    ).reset_index()

    # Profitable trader label (for Random Forest)
    trader_features["is_profitable"] = (trader_features["total_pnl"] > 0).astype(int)

    return trader_features

trader_df = engineer_features(df)
print(f"Trader-level features: {trader_df.shape}")
trader_df.head(5)

---
## 4 · ML Models

### 4a · KMeans — Trader Archetypes

In [ ]:
CLUSTER_FEATURES = ["win_rate", "avg_pnl_per_size", "contrarian_score", "trader_consistency"]

X_cluster = trader_df[CLUSTER_FEATURES].fillna(0)
scaler    = StandardScaler()
X_scaled  = scaler.fit_transform(X_cluster)

# Elbow method to validate k=4
inertia = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertia.append(km.inertia_)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(2, 9), inertia, "o-", color="#3498db", linewidth=2)
ax.set_title("Elbow Method — Optimal k", fontweight="bold")
ax.set_xlabel("Number of Clusters (k)")
ax.set_ylabel("Inertia")
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}05_elbow.png", bbox_inches="tight")
plt.show()

In [ ]:
K = 4
km_final = KMeans(n_clusters=K, random_state=42, n_init=10)
trader_df["cluster"] = km_final.fit_predict(X_scaled)

# Summarise clusters
cluster_summary = trader_df.groupby("cluster")[
    ["win_rate","avg_pnl","avg_pnl_per_size","contrarian_score","trader_consistency","trade_count"]
].mean().round(3)

# Name archetypes based on cluster characteristics
def name_cluster(row):
    if row["win_rate"] > 0.55 and row["avg_pnl"] > 0:
        return "Consistent Performers"
    elif row["contrarian_score"] > 0.5 and row["avg_pnl"] > 0:
        return "Contrarian Winners"
    elif row["win_rate"] < 0.45 and row["avg_pnl"] < 0:
        return "Struggling Traders"
    else:
        return "Sentiment Followers"

cluster_summary["archetype"] = cluster_summary.apply(name_cluster, axis=1)
print(cluster_summary[["archetype","win_rate","avg_pnl","contrarian_score","trader_consistency"]])

In [ ]:
archetype_map = cluster_summary["archetype"].to_dict()
trader_df["archetype"] = trader_df["cluster"].map(archetype_map)

fig, ax = plt.subplots(figsize=(9, 6))
for arch, grp in trader_df.groupby("archetype"):
    ax.scatter(grp["win_rate"], grp["avg_pnl"].clip(-200, 200),
               label=arch, alpha=0.7, s=50, edgecolors="white", linewidth=0.4)

ax.axhline(0,  color="black", linewidth=0.7, linestyle="--", alpha=0.4)
ax.axvline(0.5,color="black", linewidth=0.7, linestyle="--", alpha=0.4)
ax.set_title("Trader Archetypes (KMeans k=4)", fontweight="bold", pad=14)
ax.set_xlabel("Win Rate"); ax.set_ylabel("Avg PnL per Trade (clipped ±200 USD)")
ax.legend(title="Archetype", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}06_archetypes_scatter.png", bbox_inches="tight")
plt.show()
print("   trade against the crowd and still book positive PnL — the rarest archetype.")

### 4b · Random Forest — What Drives Profitability?

In [ ]:
RF_FEATURES = ["win_rate","avg_pnl_per_size","contrarian_score",
               "trader_consistency","trade_count","sentiment_score"]

X_rf = trader_df[RF_FEATURES].fillna(0)
y_rf = trader_df["is_profitable"]

X_train, X_test, y_train, y_test = train_test_split(
    X_rf, y_rf, test_size=0.25, random_state=42, stratify=y_rf
)

rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42, class_weight="balanced")
rf.fit(X_train, y_train)

print("── Classification Report ──")
print(classification_report(y_test, rf.predict(X_test), target_names=["Unprofitable","Profitable"]))

In [ ]:
importances = pd.Series(rf.feature_importances_, index=RF_FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(importances.index, importances.values,
               color=["#2ecc71" if v > importances.median() else "#bdc3c7" for v in importances.values],
               edgecolor="white", alpha=0.88)
for bar, val in zip(bars, importances.values):
    ax.text(val+0.002, bar.get_y()+bar.get_height()/2,
            f"{val:.3f}", va="center", fontsize=9)

ax.set_title("Random Forest — Feature Importance\n(What Drives Trader Profitability?)",
             fontweight="bold", pad=14)
ax.set_xlabel("Importance Score")
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}07_feature_importance.png", bbox_inches="tight")
plt.show()
print("   profitability — disciplined, consistent trading matters more than timing.")

In [ ]:
explainer   = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)

# For binary classification shap_values is a list [class0, class1]
sv = shap_values[1] if isinstance(shap_values, list) else shap_values

plt.figure(figsize=(9, 5))
shap.summary_plot(sv, X_test, feature_names=RF_FEATURES, show=False, plot_size=None)
plt.title("SHAP Summary — Impact on Profitability Prediction", fontweight="bold", pad=14)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}08_shap_summary.png", bbox_inches="tight")
plt.show()
print("   high win_rate always pushes toward profitable; high sentiment_score (Greed)")
print("   has mixed effects depending on trader type.")

---
## 5 · Executive Insights



In [ ]:
import json, subprocess

stats = {
    "avg_pnl_by_sentiment": df.groupby("sentiment")["closed_pnl"].mean().round(2).to_dict(),
    "win_rate_by_sentiment": (
        df.groupby("sentiment")["closed_pnl"]
          .apply(lambda x: round((x>0).mean()*100, 1))
          .to_dict()
    ),
    "archetype_counts": trader_df["archetype"].value_counts().to_dict(),
    "top_features": importances.tail(3).index.tolist(),
    "total_traders": int(trader_df.shape[0]),
    "total_trades":  int(len(df)),
}

# Save for insight_engine
os.makedirs("outputs", exist_ok=True)
with open("outputs/stats_summary.json","w") as f:
    json.dump(stats, f, indent=2)

print("Stats snapshot:")
print(json.dumps(stats, indent=2))

In [ ]:
!pip uninstall mistralai -y
!pip uninstall mistral-common -y
!pip install --no-cache-dir mistralai==1.5.1

In [ ]:
import mistralai


import os, json
from mistralai import Mistral

MISTRAL_API_KEY = " secret key"  

# Build a rich context object from everything computed above
context = {
    "dataset": {
        "total_trades": int(len(df)),
        "total_traders": int(trader_df.shape[0]),
        "date_range": f"{df['date'].min().date()} to {df['date'].max().date()}",
        "assets_traded": df["symbol"].nunique(),
    },
    "sentiment_analysis": {
        "avg_pnl_by_sentiment": df.groupby("sentiment")["closed_pnl"].mean().round(2).to_dict(),
        "win_rate_by_sentiment": (
            df.groupby("sentiment")["closed_pnl"]
              .apply(lambda x: round((x > 0).mean() * 100, 1))
              .to_dict()
        ),
        "trade_count_by_sentiment": df["sentiment"].value_counts().to_dict(),
    },
    "trader_archetypes": {
        "archetype_counts": trader_df["archetype"].value_counts().to_dict(),
        "archetype_avg_stats": (
            trader_df.groupby("archetype")[
                ["win_rate", "avg_pnl", "contrarian_score", "trader_consistency", "trade_count"]
            ].mean().round(3).to_dict()
        ),
    },
    "profitability_drivers": {
        "top_3_features_by_importance": importances.sort_values(ascending=False).head(3).round(4).to_dict(),
        "pct_profitable_traders": round((trader_df["is_profitable"].mean() * 100), 1),
    },
    "contrarian_behaviour": {
        "avg_contrarian_score": round(trader_df["contrarian_score"].mean(), 3),
        "contrarian_score_profitable_traders": round(
            trader_df[trader_df["is_profitable"] == 1]["contrarian_score"].mean(), 3
        ),
        "contrarian_score_unprofitable_traders": round(
            trader_df[trader_df["is_profitable"] == 0]["contrarian_score"].mean(), 3
        ),
    },
    "top_vs_bottom_traders": {
        "top_10pct_avg_pnl": round(
            trader_df[trader_df["total_pnl"] >= trader_df["total_pnl"].quantile(0.9)]["avg_pnl"].mean(), 2
        ),
        "bottom_10pct_avg_pnl": round(
            trader_df[trader_df["total_pnl"] <= trader_df["total_pnl"].quantile(0.1)]["avg_pnl"].mean(), 2
        ),
        "top_10pct_win_rate": round(
            trader_df[trader_df["total_pnl"] >= trader_df["total_pnl"].quantile(0.9)]["win_rate"].mean(), 3
        ),
        "bottom_10pct_win_rate": round(
            trader_df[trader_df["total_pnl"] <= trader_df["total_pnl"].quantile(0.1)]["win_rate"].mean(), 3
        ),
    },
}

print(json.dumps(context, indent=2))

In [ ]:
SYSTEM_PROMPT = """You are a quantitative trading analyst reviewing a statistical study of crypto trader behaviour.
You will receive a JSON object containing aggregated statistics from the analysis.
Your job is to identify non-obvious patterns, contradictions, and actionable insights that a trader or researcher would find valuable.
Do not state the obvious. Focus on what is surprising, counter-intuitive, or underexplored.

Respond strictly as a JSON object with this structure:
{
  "key_findings": [
    {
      "finding": "concise description of the pattern",
      "why_it_matters": "implication for trading or research",
      "confidence": "high / medium / low"
    }
  ],
  "contradictions": [
    "any data points that conflict with conventional wisdom or each other"
  ],
  "recommended_next_analyses": [
    "specific follow-up analyses worth running on this dataset"
  ],
  "overall_summary": "2-3 sentence synthesis"
}

Return only valid JSON. No markdown, no preamble."""

USER_PROMPT = f"""Here is the full statistical context from the analysis:

{json.dumps(context, indent=2)}

Analyse this and return your structured insights as JSON."""

client = Mistral(api_key=MISTRAL_API_KEY)

response = client.chat.complete(
    model="mistral-small-latest",   # free tier model
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": USER_PROMPT},
    ],
    temperature=0.3,
    max_tokens=1500,
)

raw = response.choices[0].message.content.strip()

# Strip markdown fences if model wraps in them
if raw.startswith("```"):
    raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()

insights = json.loads(raw)

# Save for reference
os.makedirs("outputs", exist_ok=True)
with open("outputs/llm_insights.json", "w") as f:
    json.dump(insights, f, indent=2)

print("Insights saved to outputs/llm_insights.json")

In [ ]:
CONF_COLOR = {"high": "\033[92m", "medium": "\033[93m", "low": "\033[91m", "reset": "\033[0m"}

print("=" * 65)
print("  KEY FINDINGS")
print("=" * 65)
for i, item in enumerate(insights.get("key_findings", []), 1):
    conf  = item.get("confidence", "").lower()
    color = CONF_COLOR.get(conf, "")
    print(f"\n{i}. {item['finding']}")
    print(f"   Why it matters : {item['why_it_matters']}")
    print(f"   Confidence     : {color}{conf.upper()}{CONF_COLOR['reset']}")

print("\n" + "-" * 65)
print("  CONTRADICTIONS")
print("-" * 65)
for c in insights.get("contradictions", []):
    print(f"  - {c}")

print("\n" + "-" * 65)
print("  RECOMMENDED NEXT ANALYSES")
print("-" * 65)
for r in insights.get("recommended_next_analyses", []):
    print(f"  - {r}")

print("\n" + "-" * 65)
print("  SUMMARY")
print("-" * 65)
print(insights.get("overall_summary", ""))
print("=" * 65)

##  Summary

This analysis examined **Hyperliquid historical trade data** merged with the **Bitcoin Fear & Greed Index** to uncover how market sentiment influences trader behaviour and profitability.

---

###  Dataset Overview
| Metric | Value |
|---|---|
| Source | Hyperliquid trades × Bitcoin Fear/Greed Index |
| Pipeline | Merge on date → Feature engineering → ML models → LLM insights |

---

###  Sentiment & Performance
- **PnL spreads widen sharply during Extreme Greed** — both the largest gains and worst losses concentrate in euphoric market conditions.
- **Win rates exceed 50% across most sentiment regimes**, with Fear periods historically offering relatively safer trading windows.
- **Average PnL per trade varies by sentiment**, revealing differences in expected value depending on the prevailing market mood.

---

###  Trader Archetypes (KMeans, k=4)
Four behavioural clusters were identified:

| Archetype | Description |
|---|---|
| **Consistent Performers** | High win rate (>55%), positive average PnL across all conditions |
| **Contrarian Winners** | Trade against crowd sentiment and remain profitable — the rarest group |
| **Sentiment Followers** | Performance closely tracks prevailing market mood |
| **Struggling Traders** | Below-average win rate (<45%) and negative expected PnL |

---

###  Profitability Drivers (Random Forest + SHAP)
The top features driving trader profitability were:
1. **`win_rate`** — the single strongest predictor; disciplined trade selection dominates.
2. **`avg_pnl_per_size`** — risk-adjusted return; efficient use of capital matters.
3. **`trader_consistency`** — lower PnL variance consistently outperforms high-variance strategies.

> **Key insight:** Discipline and consistency outweigh market timing as determinants of profitability.

---

###  Top 10% vs Bottom 10% Traders
- Top-decile traders maintain **higher win rates across all sentiment regimes**, suggesting their edge is structural rather than luck-driven.
- Bottom-decile traders show **much wider performance variance**, particularly during Extreme Greed.

---

###  LLM-Augmented Insights (Mistral)
An LLM layer synthesised the quantitative results into:
- **Key findings** with confidence scores (High / Medium / Low)
- **Contradictions** vs conventional trading wisdom
- **Recommended follow-up analyses** for deeper investigation

---

